# Étape 18 — Robustesse des résultats sur les graphes appris (Junyi)

Deux demandes de l'encadrant avant la rédaction définitive :

1. **Mesures complémentaires** : une AUC de 0,54 peut coexister avec une précision faible ou élevée selon la rareté des vrais prérequis (3,6 % des paires candidates).
   On ajoute l'**AUPRC** (aire sous la courbe précision-rappel ; hasard = part des vrais prérequis) et des mesures **top-K** (précision et rappel sur les K paires les mieux classées).
   Cellule R1 : calculé **sans réentraîner**, à partir des poids enregistrés à l'étape 15.
2. **Sensibilité aux réglages** : la conclusion « le graphe appris retrouve peu les prérequis » ne doit pas venir d'un mauvais réglage des pénalités.
   Cellule R2 : 9 combinaisons de pénalités (parcimonie L1 ∈ {0 ; 0,01 ; 0,1} × acyclicité NOTEARS ∈ {0 ; 0,01 ; 0,1}), plus un poids de départ de 0,5, plus un **autre ensemble de paires candidates** (même domaine `area`, plus large que le thème). 100 % des élèves, 3 graines.

**Règle de choix fixée à l'avance (pas de sélection sur le test)** : le réglage retenu est celui qui a la meilleure **AUC de validation du KT** ; on rapporte sa récupération des prérequis.
On rapporte aussi, à titre de borne haute, la **meilleure récupération obtenue par n'importe quel réglage** : si même ce maximum reste faible, la conclusion ne dépend pas du réglage.

**Deux comptes en parallèle :** `PARTIE = 'A'` (graines 0 et 1) et `PARTIE = 'B'` (graine 2). Environ 4 à 6 min par entraînement, 11 réglages par graine (A ≈ 2 h, B ≈ 1 h). **GPU T4.**

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2, lr=1e-3, batch=64, epoques_max=60, patience=5)
GRAINE = 42

In [ ]:
# Cellule 2 — Séquences, graphe expert, et repérage des PREMIÈRES tentatives
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv')).sort_values(['uid', 'morceau'])
    U, Q, R, F = [], [], [], []
    vus = {}
    for uid, ex, rep in zip(df['uid'], df['exercices'], df['reponses']):
        q = np.array(ex.split(','), dtype=np.int64)
        r = np.array(rep.split(','), dtype=np.int64)
        deja = vus.setdefault(uid, set())          # exercices déjà tentés par cet élève (tous morceaux confondus)
        f = np.zeros(len(q), dtype=bool)
        for i, x in enumerate(q):
            if x not in deja:
                f[i] = True
                deja.add(x)
        U.append(uid); Q.append(q); R.append(r); F.append(f)
    return np.array(U), Q, R, F

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r, f) in donnees.items():
    n_rep = sum(map(len, q)); n_first = sum(int(x.sum()) for x in f)
    print(f'{p:5s} : {len(set(u)):6,} élèves | {n_rep:9,} réponses | premières tentatives : {n_first:8,} ({n_first/n_rep:.1%})')

In [ ]:
# Cellule 3 — Tenseurs (Q exercices, R réponses, M masque, F première tentative)
def tenseurs(q_list, r_list, f_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64); R = np.zeros_like(Q)
    M = np.zeros((len(q_list), L), dtype=bool);     F = np.zeros_like(M)
    for i, (q, r, f) in enumerate(zip(q_list, r_list, f_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)], F[i, :len(f)] = q, r, True, f
    return tuple(torch.from_numpy(x) for x in (Q, R, M, F))

def sous_ensemble_train(fraction, graine=GRAINE):
    u, q, r, f = donnees['train']
    if fraction >= 1:
        return q, r, f
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    gardes = set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx], [f[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('OK')

### Les modèles

- **GraphKT** : identique à l'étape 5 (le graphe lisse les représentations des exercices, avec un canal « prérequis » et un canal « successeurs »).
- **PrereqKT** : GraphKT **plus** 6 indicateurs calculés à chaque instant pour l'exercice à prédire :
  taux de réussite moyen de l'élève sur ses **prérequis** déjà tentés, part des prérequis déjà tentés, présence de prérequis, et les trois mêmes indicateurs pour ses **successeurs**.
  C'est la traduction directe de l'hypothèse « *s'il a échoué sur A, il échouera plus souvent sur B* ».
  Avec un graphe vide, ces indicateurs valent 0 : PrereqKT redevient GraphKT sans graphe.

In [ ]:
# Cellule 4 — Modèles
def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)

class GraphKT(nn.Module):
    def __init__(self, n, hp, A, indicateurs=False):
        super().__init__()
        d, h = hp['dim'], hp['hidden']
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : successeurs de i
        self.n, self.indicateurs = n, indicateurs
        self.E = nn.Embedding(n, d)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, h, batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        n_ind = 6 if indicateurs else 0
        self.pred = nn.Sequential(nn.Linear(h + d + n_ind, h), nn.ReLU(), nn.Linear(h, 1))

    def representations(self):
        H = self.E.weight
        a_pre = (self.P_pre.sum(1, keepdim=True) > 0).float()
        a_suc = (self.P_suc.sum(1, keepdim=True) > 0).float()
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * a_pre + torch.relu(c['suc'](self.P_suc @ H)) * a_suc
        return H

    def calcul_indicateurs(self, q, r):
        # Historique cumulé par exercice jusqu'à l'instant t : nombre d'essais et de réussites (B, L-1, N)
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float()
        taux = reussites / essais.clamp(min=1)
        sortie = []
        for P in (self.P_pre, self.P_suc):
            voisins = (P[q[:, 1:]] > 0).float()                 # voisins de l'exercice à prédire (B, L-1, N)
            n_vois = voisins.sum(-1)
            n_tentes = (voisins * tente).sum(-1)
            moy = (voisins * tente * taux).sum(-1) / n_tentes.clamp(min=1)
            sortie += [moy, n_tentes / n_vois.clamp(min=1), (n_vois > 0).float()]
        return torch.stack(sortie, -1)

    def forward(self, q, r):
        H = self.representations()
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        x = [self.drop(h), H[q[:, 1:]]]
        if self.indicateurs:
            x.append(self.calcul_indicateurs(q, r))
        return self.pred(torch.cat(x, dim=-1)).squeeze(-1)

print('OK')

In [ ]:
# Cellule 5 — Graphes : expert, aucun, aléatoire, et corruption contrôlée E1–E5
import networkx as nx

def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def corrompre(aretes, type_err, taux, graine=GRAINE, topics=exos['topic'].values):
    rng = random.Random(10_000 + graine)
    E = set(map(tuple, aretes))
    k = int(round(taux * len(E)))
    noeuds = list(range(N))
    if type_err == 'E1':      # fausses flèches aléatoires
        while k > 0:
            a, b = rng.sample(noeuds, 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E2':    # fausses flèches plausibles : même thème
        par_topic = {}
        for i, t in enumerate(topics):
            par_topic.setdefault(t, []).append(i)
        groupes = [g for g in par_topic.values() if len(g) >= 2]
        essais = 0
        while k > 0 and essais < 100 * len(E) + 1000:
            essais += 1
            a, b = rng.sample(rng.choice(groupes), 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E3':    # inversions
        for a, b in rng.sample(sorted(E), min(k, len(E))):
            E.discard((a, b)); E.add((b, a))
    elif type_err == 'E4':    # suppressions
        for e in rng.sample(sorted(E), min(k, len(E))):
            E.discard(e)
    elif type_err == 'E5':    # cycles : C -> A alors que A mène déjà à C
        G = nx.DiGraph(list(E))
        candidats = sorted(E); rng.shuffle(candidats)
        for a, b in candidats:
            if k == 0:
                break
            c = rng.choice(list(nx.descendants(G, b)) + [b])
            if c != a and (c, a) not in E:
                E.add((c, a)); G.add_edge(c, a); k -= 1
    return sorted(E)

def graphe(nom, graine=GRAINE, taux=0.0):
    if nom == 'expert':
        return A_expert
    if nom == 'aucun':
        return np.zeros((N, N), dtype=np.float32)
    if nom == 'aleatoire':
        rng = random.Random(graine)
        E = set()
        while len(E) < len(aretes_expert):
            a, b = rng.sample(range(N), 2)
            E.add((a, b))
        return vers_matrice(E)
    return vers_matrice(corrompre(aretes_expert, nom, taux, graine))

def f1_aretes(A, B=A_expert):
    P, V = set(zip(*np.nonzero(A))), set(zip(*np.nonzero(B)))
    tp = len(P & V); p = tp / max(1, len(P)); r = tp / max(1, len(V))
    return round(2 * p * r / (p + r), 3) if p + r else 0.0

print(f'{"graphe":8s} {"taux":>5s} {"flèches":>8s} {"F1/expert":>10s} {"acyclique":>10s}')
for t in ['E1', 'E2', 'E3', 'E4', 'E5']:
    for taux in [0.1, 0.3, 0.5]:
        A = graphe(t, 0, taux)
        print(f'{t:8s} {taux:5.0%} {int(A.sum()):8d} {f1_aretes(A):10.3f} {str(nx.is_directed_acyclic_graph(nx.DiGraph(A))):>10s}')

In [ ]:
# Cellule 6 — Entraînement et évaluation par sous-ensemble
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=128):
    modele.eval()
    Q, R, M, F = T
    ys, ps, fs = [], [], []
    for i in range(0, len(Q), batch):
        q, r, m, f = (x[i:i+batch].to(DEVICE) for x in (Q, R, M, F))
        p = torch.sigmoid(modele(q, r))
        c = m[:, 1:]
        ys.append(r[:, 1:][c].cpu().numpy()); ps.append(p[c].float().cpu().numpy()); fs.append(f[:, 1:][c].cpu().numpy())
    y, p, f = (np.concatenate(x) for x in (ys, ps, fs))
    auc = lambda s: round(float(roc_auc_score(y[s], p[s])), 4) if len(set(y[s])) == 2 else None
    tout = np.ones_like(f, dtype=bool)
    return {'AUC': auc(tout), 'AUC_1re': auc(f), 'AUC_rep': auc(~f), 'ACC': round(float(accuracy_score(y, p >= 0.5)), 4)}

def entrainer(nom_modele, A, fraction=1.0, graine=GRAINE, hp=HP):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = GraphKT(N, hp, A, indicateurs=(nom_modele == 'PrereqKT')).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        # print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    return {'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
            'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1)}
print('OK')

In [ ]:
# Cellule L1 — Paires candidates (même thème), score du LLM, vrais prérequis, ordre de passage des élèves
from collections import Counter
from itertools import permutations

scores = pd.read_csv(os.path.join(RESULTATS, 'junyi_llm_scores.csv'))
scores[['A', 'B']] = scores[['A', 'B']].astype(int)
groupes = exos.dropna(subset=['topic']).groupby('topic')['exo_id'].apply(list)
cand = pd.DataFrame([(a, b) for ids in groupes for a in ids for b in ids if a != b], columns=['A', 'B'])
cand = cand.merge(scores[['A', 'B', 'p_oui']].drop_duplicates(['A', 'B']), on=['A', 'B'], how='left')
manque = int(cand.p_oui.isna().sum())
if manque:
    print(f'Attention : {manque} paires sans score LLM, remplacées par la médiane')
    cand['p_oui'] = cand.p_oui.fillna(scores.p_oui.median())
PAIRES = cand[['A', 'B']].values.astype(np.int64)

G_exp = nx.DiGraph(); G_exp.add_nodes_from(range(N)); G_exp.add_edges_from(map(tuple, aretes_expert))
anc = {b: nx.ancestors(G_exp, b) for b in range(N)}
cand['expert'] = A_expert[cand.A, cand.B] > 0
cand['ancetre'] = [a in anc[b] for a, b in PAIRES]
topic_de = exos.set_index('exo_id')['topic'].to_dict()

def eleves_choisis(fraction, graine):          # même tirage que sous_ensemble_train
    u = donnees['train'][0]
    if fraction >= 1:
        return set(u)
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    return set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))

def ordre_passage(fraction, graine):
    # Pour chaque paire (A, B) : nombre d'élèves ayant fait A et B, et nombre de ceux qui ont fait A en premier (1re tentative)
    u, q, _, f = donnees['train']
    gardes = eleves_choisis(fraction, graine)
    rang, compteur = {}, {}
    for uid, qq, ff in zip(u, q, f):
        if uid not in gardes:
            continue
        d = rang.setdefault(uid, {}); k = compteur.get(uid, 0)
        for x, fl in zip(qq, ff):
            if fl:
                d[int(x)] = k
            k += 1
        compteur[uid] = k
    deux, avant = Counter(), Counter()
    for d in rang.values():
        par_theme = {}
        for x in d:
            t = topic_de.get(x)
            if isinstance(t, str):
                par_theme.setdefault(t, []).append(x)
        for xs in par_theme.values():
            for a, b in permutations(xs, 2):
                deux[(a, b)] += 1
                if d[a] < d[b]:
                    avant[(a, b)] += 1
    n2 = np.array([deux[(a, b)] for a, b in PAIRES]); na = np.array([avant[(a, b)] for a, b in PAIRES])
    return n2, na

n2, na = ordre_passage(1.0, 0)                 # description du programme : tous les élèves d'entraînement
part_avant = na / np.maximum(n2, 1)
cand['n_eleves_deux'] = n2
cand['ordre'] = np.where(n2 < 20, 'rare', np.where((part_avant >= 0.95) | (part_avant <= 0.05), 'impose', 'variable'))
SOUS_ENS = {'toutes': np.ones(len(cand), bool), 'ordre_impose': (cand.ordre == 'impose').values,
            'ordre_variable': (cand.ordre == 'variable').values}

print(f'{len(cand):,} paires candidates | flèches expertes parmi elles : {int(cand.expert.sum())} '
      f'| prérequis directs ou indirects : {int(cand.ancetre.sum())}')
print(cand.groupby('ordre').agg(paires=('A', 'size'), dont_expertes=('expert', 'sum')).to_string())

def qualite(score, prefixe):
    # AUC face aux flèches expertes (directes), AUC face aux prérequis directs ou indirects, précision@K, par sous-ensemble
    score = np.asarray(score, dtype=float); res = {}
    for nom, m in SOUS_ENS.items():
        y, ya, s = cand.expert.values[m], cand.ancetre.values[m], score[m]
        ok = len(set(y)) == 2
        res[f'{prefixe}AUC_{nom}'] = round(roc_auc_score(y, s), 4) if ok else None
        res[f'{prefixe}AUC_etendue_{nom}'] = round(roc_auc_score(ya, s), 4) if len(set(ya)) == 2 else None
        K = int(y.sum())
        res[f'{prefixe}precisionK_{nom}'] = round(float(y[np.argsort(-s, kind='stable')[:K]].mean()), 4) if K else None
    return res

ref_llm = qualite(cand.p_oui.values, '')
print('\nScore du LLM (départ de prior_llm) :', {k: v for k, v in ref_llm.items() if 'AUC_' in k and 'etendue' not in k})

In [ ]:
# Cellule L2 — Le modèle à flèches apprises (identique à l'étape 15)
HP_APPRIS = dict(HP, lam_l1=0.01, lam_acyc=0.01, lr_aretes=1e-2, poids_depart=0.1)   # lam_acyc faible : avec le prior LLM, la pénalité de départ est grande

class PrereqKTAppris(GraphKT):
    def __init__(self, n, hp, paires, logits_depart):
        super().__init__(n, hp, np.zeros((n, n), np.float32), indicateurs=True)
        self.register_buffer('src', torch.as_tensor(paires[:, 0]))
        self.register_buffer('dst', torch.as_tensor(paires[:, 1]))
        self.theta = nn.Parameter(torch.as_tensor(logits_depart, dtype=torch.float32))

    def poids(self):
        return torch.sigmoid(self.theta)

    def W(self):                                   # W[a, b] = poids de la flèche a -> b
        W = torch.zeros(self.n, self.n, device=self.theta.device)
        return W.index_put((self.src, self.dst), self.poids())

    def forward(self, q, r):
        W = self.W()
        P_pre = W.T / (W.T.sum(1, keepdim=True) + 1e-6)     # ligne j : prérequis de j (pondérés)
        P_suc = W / (W.sum(1, keepdim=True) + 1e-6)         # ligne i : successeurs de i
        a_pre = 1 - torch.exp(-W.T.sum(1, keepdim=True)); a_suc = 1 - torch.exp(-W.sum(1, keepdim=True))
        H = self.E.weight
        for c in self.couches:
            H = H + torch.relu(c['pre'](P_pre @ H)) * a_pre + torch.relu(c['suc'](P_suc @ H)) * a_suc
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        return self.pred(torch.cat([self.drop(h), H[q[:, 1:]], self.indicateurs_doux(q, r, W)], dim=-1)).squeeze(-1)

    def indicateurs_doux(self, q, r, W):
        # mêmes 6 indicateurs que PrereqKT, mais chaque voisin compte pour le poids de sa flèche
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float(); taux = reussites / essais.clamp(min=1)
        sortie = []
        for M in (W.T, W):
            v = M[q[:, 1:]]
            n_v = v.sum(-1); n_t = (v * tente).sum(-1)
            sortie += [(v * tente * taux).sum(-1) / (n_t + 1e-6), n_t / (n_v + 1e-6), 1 - torch.exp(-n_v)]
        return torch.stack(sortie, -1)

    def acyclicite(self):                          # NOTEARS : tr(exp(W∘W)) − n, nul si et seulement si le graphe est sans cycle
        W = self.W()
        return torch.trace(torch.matrix_exp(W * W)) - self.n

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 0.01, 0.99)
    return np.log(p / (1 - p))

def entrainer_appris(depart, fraction, graine, hp=HP_APPRIS):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    if depart == 'sans_prior':
        l0 = logit(hp['poids_depart']) + np.random.default_rng(graine).normal(0, 0.01, len(PAIRES))
    else:
        l0 = logit(cand.p_oui.values)
    modele = PrereqKTAppris(N, hp, PAIRES, l0).to(DEVICE)
    autres = [p for n_, p in modele.named_parameters() if n_ != 'theta']
    opt = torch.optim.Adam([{'params': autres, 'lr': hp['lr']}, {'params': [modele.theta], 'lr': hp['lr_aretes']}])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float()) \
                + hp['lam_l1'] * modele.poids().mean() + hp['lam_acyc'] * modele.acyclicite()
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    with torch.no_grad():
        poids = modele.poids().cpu().numpy()
        acyc = float(modele.acyclicite())
    return ({'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
             'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1), 'acyclicite': round(acyc, 4),
             'poids_moyen': round(float(poids.mean()), 4), 'fleches_poids_sup_0.5': int((poids > 0.5).sum())}, poids)

In [ ]:
# Cellule R0 — Mesures de récupération des prérequis : AUC, AUPRC, précision et rappel top-K
from sklearn.metrics import average_precision_score

def recuperation(score, y):
    score, y = np.asarray(score, float), np.asarray(y, bool)
    ordre = np.argsort(-score, kind='stable'); K = int(y.sum())
    return {'AUC': round(roc_auc_score(y, score), 4), 'AUPRC': round(average_precision_score(y, score), 4),
            'AUPRC_hasard': round(float(y.mean()), 4),
            'precision_topK': round(float(y[ordre[:K]].mean()), 4), 'rappel_top2K': round(float(y[ordre[:2 * K]].sum() / K), 4),
            'precision_top100': round(float(y[ordre[:100]].mean()), 4)}
print('Vrais prérequis parmi les paires candidates :', int(cand.expert.sum()), f'({cand.expert.mean():.1%})')

In [ ]:
# Cellule R1 — AUPRC et top-K pour les résultats de l'étape 15 (poids enregistrés, aucun entraînement ; quelques minutes)
DOSSIER_POIDS = os.path.join(RESULTATS, 'rq3_poids_appris')
lignes = []
for frac in [1.0, 0.05]:
    for g_ in range(5):
        n2_, na_ = ordre_passage(frac, g_)
        sources = {'score_llm': cand.p_oui.values, 'ordre_passage': (na_ + 1) / (n2_ + 2)}
        for lib in ['appris_sans_prior', 'appris_prior_llm']:
            c_ = os.path.join(DOSSIER_POIDS, f'{lib}_{frac}_{g_}.npy')
            if os.path.exists(c_):
                sources[lib] = np.load(c_)
        for nom, s in sources.items():
            for sous, m in SOUS_ENS.items():
                if cand.expert.values[m].sum() > 0:
                    lignes.append({'fraction': frac, 'graine': g_, 'source': nom, 'paires': sous, **recuperation(s[m], cand.expert.values[m])})
r1 = pd.DataFrame(lignes)
r1.to_csv(os.path.join(RESULTATS, 'rq3_recuperation_auprc.csv'), index=False)
synth_r1 = r1.groupby(['fraction', 'paires', 'source'])[['AUC', 'AUPRC', 'AUPRC_hasard', 'precision_topK', 'rappel_top2K', 'precision_top100']].mean().round(4)
synth_r1.to_csv(os.path.join(RESULTATS, 'rq3_recuperation_auprc_synthese.csv'))
print(synth_r1.to_string())

In [ ]:
# Cellule R1b — Forme des graphes appris : acyclicité NOTEARS, cycles, parcimonie, distribution des poids (aucun entraînement ; 2 à 5 min)
#   Après une déconnexion : relancez les cellules 1 à 8 (jusqu'à L1 incluse), puis R0, puis celle-ci.
import networkx as nx
from scipy.linalg import expm
DOSSIER_POIDS = os.path.join(RESULTATS, 'rq3_poids_appris')
K_exp = int(cand.expert.sum())
lignes = []
for frac in [1.0, 0.05]:
    for g_ in range(5):
        for lib in ['appris_sans_prior', 'appris_prior_llm']:
            c_ = os.path.join(DOSSIER_POIDS, f'{lib}_{frac}_{g_}.npy')
            if not os.path.exists(c_):
                continue
            w = np.load(c_).astype(float)
            W = np.zeros((N, N)); W[PAIRES[:, 0], PAIRES[:, 1]] = w
            h = float(np.trace(expm(W * W)) - N)                     # pénalité NOTEARS réellement obtenue (0 = sans cycle)
            ligne = {'fraction': frac, 'graine': g_, 'source': lib, 'h_NOTEARS': round(h, 4),
                     'poids_moyen': round(w.mean(), 4), 'poids_median': round(float(np.median(w)), 4),
                     'poids_q90': round(float(np.quantile(w, 0.9)), 4), 'poids_max': round(w.max(), 4),
                     'part_poids>0.1_%': round(100 * (w > 0.1).mean(), 2), 'part_poids>0.5_%': round(100 * (w > 0.5).mean(), 2)}
            ordre_w = np.argsort(-w, kind='stable')
            graphes = {f'seuil{t}': np.where(w > t)[0] for t in (0.1, 0.3, 0.5)}
            graphes['topK'] = ordre_w[:K_exp]                         # les K paires les mieux classées (K = nombre de vrais prérequis)
            for nom, idx in graphes.items():
                G = nx.DiGraph(); G.add_nodes_from(range(N)); G.add_edges_from(map(tuple, PAIRES[idx]))
                composantes = [c for c in nx.strongly_connected_components(G) if len(c) > 1]
                ligne[f'{nom}_aretes'] = len(idx)
                ligne[f'{nom}_sans_cycle'] = nx.is_directed_acyclic_graph(G)
                ligne[f'{nom}_noeuds_dans_cycles'] = sum(len(c) for c in composantes)
                ligne[f'{nom}_cycles_2'] = sum(1 for a, b in G.edges if a < b and G.has_edge(b, a))
            lignes.append(ligne)
r1b = pd.DataFrame(lignes)
r1b.to_csv(os.path.join(RESULTATS, 'rq3_forme_graphe_appris.csv'), index=False)
synth_r1b = r1b.drop(columns='graine').groupby(['fraction', 'source']).mean(numeric_only=True).round(4)
synth_r1b.to_csv(os.path.join(RESULTATS, 'rq3_forme_graphe_appris_synthese.csv'))
print(f'K = {K_exp} vrais prérequis parmi les paires candidates')
print(synth_r1b.T.to_string())

In [ ]:
# Cellule R2 — Sensibilité du graphe appris aux pénalités et aux paires candidates (100 % des élèves ; reprise automatique)
PARTIE = 'A'          # 'A' = graines 0 et 1 ; 'B' = graine 2 (sur l'autre compte, en même temps)

GRAINES_PARTIE = {'A': [0, 1], 'B': [2]}[PARTIE]
CONFIGS = [(f'l1={l1}_acyc={ac}', dict(lam_l1=l1, lam_acyc=ac), 'topic') for l1 in [0.0, 0.01, 0.1] for ac in [0.0, 0.01, 0.1]]
CONFIGS += [('depart=0.5', dict(poids_depart=0.5), 'topic'), ('candidats=area', {}, 'area')]

# Autre ensemble de candidats : toutes les paires d'exercices du même domaine (area), plus large que le thème
grp_a = exos.dropna(subset=['area']).groupby('area')['exo_id'].apply(list)
PAIRES_AREA = np.array([(a, b) for ids in grp_a for a in ids for b in ids if a != b], dtype=np.int64)
y_area = A_expert[PAIRES_AREA[:, 0], PAIRES_AREA[:, 1]] > 0
pos_topic = {p: i for i, p in enumerate(map(tuple, PAIRES))}
idx_topic_dans_area = np.array([pos_topic.get(tuple(p), -1) for p in PAIRES_AREA])
print(f'Paires « area » : {len(PAIRES_AREA):,} (dont {int(y_area.sum())} vrais prérequis) | paires « topic » : {len(PAIRES):,}')

chemin_g = os.path.join(RESULTATS, f'rq3_sensibilite_{PARTIE}.csv')
res_g = pd.read_csv(chemin_g).to_dict('records') if os.path.exists(chemin_g) else []
fait = set()
for p_ in ['A', 'B']:
    c_ = os.path.join(RESULTATS, f'rq3_sensibilite_{p_}.csv')
    if os.path.exists(c_):
        fait |= {(x['config'], int(x['graine'])) for x in pd.read_csv(c_).to_dict('records')}
PAIRES_TOPIC = PAIRES.copy()
t0_ = time.time()
for g_ in GRAINES_PARTIE:
    for nom, reglage, candidats in CONFIGS:
        if (nom, g_) in fait:
            continue
        PAIRES = PAIRES_AREA if candidats == 'area' else PAIRES_TOPIC          # utilisé par entrainer_appris
        try:
            res, poids = entrainer_appris('sans_prior', 1.0, g_, hp=dict(HP_APPRIS, **reglage))
        finally:
            PAIRES = PAIRES_TOPIC
        if candidats == 'area':
            m_ = idx_topic_dans_area >= 0
            p_topic = np.zeros(len(PAIRES_TOPIC)); p_topic[idx_topic_dans_area[m_]] = poids[m_]
            rec = {**{f'{k}': v for k, v in recuperation(p_topic, cand.expert.values).items()},
                   **{f'{k}_toutes_paires_area': v for k, v in recuperation(poids, y_area).items()}}
        else:
            rec = recuperation(poids, cand.expert.values)
        res_g.append({'config': nom, 'candidats': candidats, 'graine': g_, **reglage, **res, **rec})
        pd.DataFrame(res_g).to_csv(chemin_g, index=False)
        print(f'  graine {g_} | {nom:18s} | AUC valid KT = {res["AUC_valid"]} | AUC 1res = {res["AUC_1re_test"]} | '
              f'récupération AUC = {rec["AUC"]} | AUPRC = {rec["AUPRC"]} | {res["minutes"]} min')
    print(f'Graine {g_} terminée ({(time.time() - t0_) / 60:.0f} min écoulées)')
print(f'Partie {PARTIE} terminée.')

In [ ]:
# Cellule R3 — Analyse de la sensibilité (quand les deux parties sont finies)
#   Après une déconnexion : relancez les cellules 1 à R0, puis celle-ci.
import matplotlib.pyplot as plt
sens = pd.concat([pd.read_csv(os.path.join(RESULTATS, f'rq3_sensibilite_{p}.csv')) for p in ['A', 'B']
                  if os.path.exists(os.path.join(RESULTATS, f'rq3_sensibilite_{p}.csv'))], ignore_index=True)
cols = ['AUC_valid', 'AUC_1re_test', 'AUC', 'AUPRC', 'precision_topK', 'rappel_top2K', 'poids_moyen', 'fleches_poids_sup_0.5', 'acyclicite']
tab_s = sens.groupby('config')[[c for c in cols if c in sens]].agg(['mean', 'std']).round(4)
tab_s.columns = [f'{a}_{b}' for a, b in tab_s.columns]
tab_s['graines'] = sens.groupby('config').size()
tab_s = tab_s.sort_values('AUC_valid_mean', ascending=False)
tab_s.to_csv(os.path.join(RESULTATS, 'rq3_sensibilite_synthese.csv'))
print(tab_s[[c for c in tab_s.columns if c.endswith('_mean')] + ['graines']].to_string())

retenu = tab_s[~tab_s.index.isin(['candidats=area'])].index[0]
meilleur = tab_s['AUC_mean'].idxmax()
hasard_auprc = round(float(cand.expert.mean()), 4)
print(f'\nRéglage retenu (meilleure AUC de validation du KT, règle fixée à l\'avance) : {retenu} '
      f'-> récupération AUC = {tab_s.at[retenu, "AUC_mean"]} | AUPRC = {tab_s.at[retenu, "AUPRC_mean"]} (hasard {hasard_auprc})')
print(f'Borne haute (meilleure récupération, tous réglages confondus) : {meilleur} -> AUC = {tab_s.at[meilleur, "AUC_mean"]} | '
      f'AUPRC = {tab_s.at[meilleur, "AUPRC_mean"]}')
r1s = pd.read_csv(os.path.join(RESULTATS, 'rq3_recuperation_auprc_synthese.csv'))
ref = r1s[(r1s.fraction == 1.0) & (r1s.paires == 'toutes')].set_index('source')
print('Références (100 %, toutes paires) :', {s: (ref.at[s, 'AUC'], ref.at[s, 'AUPRC']) for s in ref.index})

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for c, r in tab_s.iterrows():
    ax.errorbar(r['AUC_valid_mean'], r['AUC_mean'], xerr=r['AUC_valid_std'], yerr=r['AUC_std'], fmt='o', ms=6,
                color='tab:blue' if c == retenu else 'grey', capsize=2)
    ax.annotate(c, (r['AUC_valid_mean'], r['AUC_mean']), fontsize=7, xytext=(4, 4), textcoords='offset points')
for s, st in [('ordre_passage', '--'), ('score_llm', ':')]:
    if s in ref.index:
        ax.axhline(ref.at[s, 'AUC'], ls=st, c='black', lw=1, label=f'{s} ({ref.at[s, "AUC"]:.2f})')
ax.axhline(0.5, c='grey', lw=0.8, label='hasard (0,50)')
ax.set_xlabel('AUC de validation du KT (critère de choix)'); ax.set_ylabel('AUC de récupération des prérequis experts')
ax.set_title('Graphe appris : aucun réglage ne retrouve bien les prérequis'); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(os.path.join(RESULTATS, 'rq3_sensibilite.png'), dpi=200); plt.show()

In [ ]:
# Cellule R4 — Rassembler les résultats de l'étape 18 dans un zip et le télécharger
import zipfile
FICHIERS = ['rq3_recuperation_auprc.csv', 'rq3_recuperation_auprc_synthese.csv', 'rq3_forme_graphe_appris.csv',
            'rq3_forme_graphe_appris_synthese.csv', 'rq3_sensibilite_A.csv', 'rq3_sensibilite_B.csv',
            'rq3_sensibilite_synthese.csv', 'rq3_sensibilite.png']
chemin_zip = '/content/resultats_etape18.zip'
with zipfile.ZipFile(chemin_zip, 'w') as z:
    for f in FICHIERS:
        c = os.path.join(RESULTATS, f)
        if os.path.exists(c):
            z.write(c, arcname=f); print('ajouté   :', f)
        else:
            print('MANQUANT :', f)
try:
    from google.colab import files
    files.download(chemin_zip)
except ImportError:
    print('Hors de Colab : le fichier est ici ->', chemin_zip)